# Khám phá và Tiền xử lý dữ liệu (ShanghaiTech Dataset)

**Nội dung thực hiện:**
1. Thiết lập môi trường và cấu hình đường dẫn.
2. Làm sạch dữ liệu.
3. Chuyển đổi dữ liệu sang định dạng YOLOv8.
4. Khám phá thống kê (EDA).
5. Trực quan hóa mẫu dữ liệu (Visualization).



## 1. Thiết lập môi trường và đường dẫn


In [ ]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.io as sio
import PIL.Image as Image
import cv2
import yaml
from matplotlib import pyplot as plt
from matplotlib import cm as CM
from scipy.ndimage import gaussian_filter
from scipy.spatial import KDTree
from tqdm import tqdm
import random
from pathlib import Path



PROJECT_ROOT  = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_ROOT     = os.path.join(PROJECT_ROOT, "data")
SHANGHAI_ROOT = os.path.join(DATA_ROOT, "ShanghaiTech")


PART_A_TRAIN  = os.path.join(SHANGHAI_ROOT, "part_A", "train_data")
PART_A_TEST   = os.path.join(SHANGHAI_ROOT, "part_A", "test_data")
PART_B_TRAIN  = os.path.join(SHANGHAI_ROOT, "part_B", "train_data")
PART_B_TEST   = os.path.join(SHANGHAI_ROOT, "part_B", "test_data")



## 2. Làm sạch dữ liệu

In [ ]:
def clean_dataset(part="A"):
    paths = [PART_A_TRAIN, PART_A_TEST] if part == "A" else [PART_B_TRAIN, PART_B_TEST]
    print(f"\n--- Kiểm tra Part {part} ---")
    
    for p in paths:
        imgs = sorted(glob.glob(os.path.join(p, "images", "*.jpg")))
        errors = 0
        
        for img_p in imgs:
            mat_p = img_p.replace(".jpg", ".mat").replace("images", "ground-truth").replace("IMG_", "GT_IMG_")
            try:
                if not os.path.exists(mat_p): raise ValueError()
                with Image.open(img_p) as img: img.verify()
            except:
                errors += 1
                
        status = "Sạch" if errors == 0 else f"={errors} Lỗi"
        print(f" {os.path.basename(p):<10}: {len(imgs):>3} images | {status}")

clean_dataset("A")
clean_dataset("B")

## 3. Chuyển đổi dữ liệu sang định dạng YOLOv8

Chuyển đổi annotation dạng `.mat` (head points) sang định dạng YOLO (bounding box).

In [ ]:

def convert_mat_to_yolo(part_path, min_box_size=10, max_box_size=120, k=4, beta=0.3):

    part_path = Path(part_path)
    image_dir = part_path / "images"
    gt_dir = part_path / "ground-truth"
    labels_dir = part_path / "labels"
    labels_dir.mkdir(parents=True, exist_ok=True)

    img_paths = list(image_dir.glob("*.jpg"))
    print(f" Đang xử lý {len(img_paths)} ảnh tại {part_path.name}...")

    for img_path in tqdm(img_paths):
        # Lấy kích thước ảnh nhanh (không load pixel)
        with Image.open(img_path) as img:
            w, h = img.size

        # Xác định đường dẫn file liên quan
        base_name = img_path.name
        mat_name = base_name.replace(".jpg", ".mat").replace("IMG_", "GT_IMG_")
        mat_path = gt_dir / mat_name
        txt_path = labels_dir / base_name.replace(".jpg", ".txt")

        if not mat_path.exists():
            continue

        # Đọc tọa độ điểm
        mat = sio.loadmat(str(mat_path))
        try:
            points = mat["image_info"][0, 0][0, 0][0].astype(np.float32)
        except (KeyError, IndexError):
            continue
            
        num_points = len(points)
        if num_points == 0:
            txt_path.touch() # Tạo file trống nếu không có người
            continue

        # --- TỐI ƯU HÓA: TÍNH TOÁN BOX KHỐI LƯỢNG LỚN (VECTORIZED) ---
        if num_points > 1:
            tree = KDTree(points)
            actual_k = min(k, num_points)
            # Truy vấn tất cả các điểm cùng một lúc
            distances, _ = tree.query(points, k=actual_k)
            
            # Lấy trung bình khoảng cách (bỏ qua cột đầu tiên vì dist = 0 với chính nó)
            if actual_k > 1:
                mean_dist = np.mean(distances[:, 1:], axis=1)
                box_sizes = mean_dist * beta
            else:
                box_sizes = np.full(num_points, min_box_size)
        else:
            box_sizes = np.full(num_points, min_box_size)

        # Ràng buộc kích thước
        box_sizes = np.clip(box_sizes, min_box_size, max_box_size)

        # Chuẩn hóa tọa độ YOLO [0, 1] cho toàn bộ mảng
        x_centers = points[:, 0] / w
        y_centers = points[:, 1] / h
        box_ws = box_sizes / w
        box_hs = box_sizes / h

        # Ghi file nhanh bằng join()
        lines = [f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}" 
                 for xc, yc, bw, bh in zip(x_centers, y_centers, box_ws, box_hs)]
        
        with open(txt_path, "w") as f:
            f.write("\n".join(lines) + "\n")

def create_yaml_optimized(data_root, filename='shanghaitech.yaml'):
    """
    Tạo file cấu trúc YAML chuẩn cho YOLOv8/v11.
    """
    data_root = Path(data_root).absolute()
    yaml_content = {
        'path': str(data_root),
        'train': 'part_A/train_data/images', # Có thể thêm list nếu có cả Part B
        'val': 'part_A/test_data/images',
        'nc': 1,
        'names': ['head']
    }
    
    yaml_path = data_root / filename
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, sort_keys=False)
    print(f" Đã tạo file cấu hình tại: {yaml_path}")

# --- THỰC THI ---
# SHANGHAI_ROOT = "Đường/dẫn/đến/ShanghaiTech"
for split in ["train", "test"]:
    path = os.path.join(SHANGHAI_ROOT, "part_A", f"{split}_data")
    if os.path.exists(path): 
        convert_mat_to_yolo(path)

create_yaml_optimized(SHANGHAI_ROOT)


## 4. Khám phá thống kê (EDA)

In [ ]:
def get_stats(path):
    img_paths = glob.glob(os.path.join(path, "images", "*.jpg"))
    counts, dims = [], []
    
    for img_p in img_paths:
        mat_p = img_p.replace(".jpg", ".mat").replace("images", "ground-truth").replace("IMG_", "GT_IMG_")
        if os.path.exists(mat_p):
            mat = sio.loadmat(mat_p)
            # Lấy số lượng người từ file .mat
            counts.append(len(mat["image_info"][0, 0][0, 0][0]))
            with Image.open(img_p) as img:
                dims.append(img.size)
                
    return pd.DataFrame({
        "Số người": counts,
        "Chiều rộng": [d[0] for d in dims],
        "Chiều cao": [d[1] for d in dims]
    })

# Danh sách các tập dữ liệu cần khám phá
dataset_splits = [
    ("Part A", "Train", PART_A_TRAIN),
    ("Part A", "Test", PART_A_TEST),
    ("Part B", "Train", PART_B_TRAIN),
    ("Part B", "Test", PART_B_TEST)
]

for part, split, path in dataset_splits:
    try:
        df = get_stats(path)
        print(f"\n{'='*20} Thống kê {part} - {split} {'='*20}")
        display(df.describe()) # Hiển thị bảng thống kê chuẩn trong Notebook
        
        # Vẽ biểu đồ phân bổ
        plt.figure(figsize=(10, 4))
        sns.histplot(df["Số người"], kde=True, color='darkcyan', bins=30)
        plt.title(f"Phân bổ số lượng người - {part} ({split})", fontsize=14, fontweight='bold')
        plt.xlabel("Số người trong một ảnh")
        plt.ylabel("Số lượng ảnh")
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Lỗi khi xử lý {part} {split}: {e}")

## 5.Trực quan hóa dữ liệu

In [ ]:


def visualize_with_ground_truth(part_path, num_samples=2):
    """
    Hiện thị so sánh và đối chiếu số lượng người từ file .mat gốc vs file .txt YOLO.
    """
    part_path = Path(part_path)
    image_dir = part_path / "images"
    gt_dir = part_path / "ground-truth"
    labels_dir = part_path / "labels"
    
    img_paths = list(image_dir.glob("*.jpg"))
    samples = random.sample(img_paths, min(num_samples, len(img_paths)))
    
    for img_path in samples:
        # 1. Lấy số lượng người gốc từ file .mat
        mat_name = img_path.name.replace(".jpg", ".mat").replace("IMG_", "GT_IMG_")
        mat_path = gt_dir / mat_name
        
        orig_count = 0
        if mat_path.exists():
            mat = sio.loadmat(str(mat_path))
            # Lấy số hàng của mảng tọa độ = số người
            orig_count = len(mat["image_info"][0, 0][0, 0][0])
        
        # 2. Lấy số lượng người từ file .txt YOLO vừa tạo
        label_path = labels_dir / img_path.name.replace(".jpg", ".txt")
        yolo_count = 0
        boxes = []
        if label_path.exists():
            with open(label_path, "r") as f:
                lines = f.readlines()
                yolo_count = len(lines)
                boxes = lines

        # 3. Vẽ ảnh
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        img_anno = img.copy()

        for line in boxes:
            parts = line.split()
            if len(parts) == 5:
                # Vẽ Box (như cũ)
                _, xc, yc, bw, bh = map(float, parts)
                x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
                x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
                cv2.rectangle(img_anno, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # 4. Hiển thị
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        
        # Ảnh gốc + Thông tin gốc
        axes[0].imshow(img)
        axes[0].set_title(f"ẢNH GỐC\nSố người thực tế (trong .mat): {orig_count}", fontsize=14, color='blue', fontweight='bold')
        axes[0].axis('off')
        
        # Ảnh nhãn + Thông tin chuyển đổi
        axes[1].imshow(img_anno)
        axes[1].set_title(f"NHÃN YOLO\nSố người chuyển sang .txt: {yolo_count}", fontsize=14, color='darkgreen', fontweight='bold')
        axes[1].axis('off')
        
        # Kiểm tra khớp dữ liệu
        if orig_count == yolo_count:
            print(f"Khớp dữ liệu cho ảnh {img_path.name}: {orig_count} người.")
        else:
            print(f" Cảnh báo: Lệch dữ liệu ở ảnh {img_path.name}! (.mat: {orig_count} vs .txt: {yolo_count})")
            
        plt.tight_layout()
        plt.show()

# --- CHẠY ---
# Truyền vào đường dẫn thư mục bộ data (ví dụ train_data)
path_to_check = os.path.join(SHANGHAI_ROOT, "part_A/train_data")
visualize_with_ground_truth(path_to_check, num_samples=2)
